# Week 5 - Apache Spark Assignment

**Name:** Mohit Poonia

## Objective
Understand Spark fundamentals and perform data cleaning, transformation and aggregation using DataFrames.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Week5 Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [2]:
df = spark.read.csv(
    r"C:\Users\Mohit\Downloads\week5_dataset.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|         Fashion|        300| Delhi| 19|       Basic|    NULL|300.0|       1|user2@gmail.com|   priya|2025-01-02 11:00:00|
|    103|      2025-01-03|  West|     Electronics|        700|Mu

In [3]:
df.printSchema()


root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [4]:
df.count()

10

## Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce stores intermediate results on disk after every stage of processing, making it slower for large-scale analytics. It is inefficient for iterative tasks such as machine learning because data must be repeatedly read from and written to disk. Apache Spark overcomes these limitations through in-memory processing, faster execution, and support for advanced analytics libraries.

## Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark stores intermediate data in memory (RAM) instead of writing it to disk after every operation. Machine learning algorithms repeatedly process the same dataset during training. By keeping data in memory, Spark avoids repeated disk I/O operations, resulting in significantly faster execution compared to traditional disk-based systems such as MapReduce.

## Q3. Remove duplicate rows based on user_id and transaction_date.

In [5]:
df_q3 = df.dropDuplicates(["user_id", "transaction_date"])

df_q3.show()

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|         Fashion|        300| Delhi| 19|       Basic|    NULL|300.0|       1|user2@gmail.com|   priya|2025-01-02 11:00:00|
|    103|      2025-01-03|  West|     Electronics|        700|Mumbai| 28|     Premium|  Active| NULL|       2|user3@gmail.com|    amit|2025-01-03 12:00:00|
|    104|      2025-01-04| North|         Grocery|        200|Ja

In [6]:
df_q3.count()

9

### Insight

Duplicate records were removed to improve data quality and prevent incorrect analytical results caused by repeated transactions.

## Q4. Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [7]:
df.filter(df.region == "West") \
  .groupBy("product_category") \
  .avg("sale_amount") \
  .show()

+----------------+----------------+
|product_category|avg(sale_amount)|
+----------------+----------------+
|         Fashion|           400.0|
|         Grocery|           350.0|
|     Electronics|           625.0|
+----------------+----------------+



### Insight

This analysis helps identify the average sales amount for each product category in the West region.

## Q5. What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

.na.drop() removes rows containing null values, while .na.fill() replaces null values with a specified value. Using .na.fill() helps preserve records while handling missing data.

In [8]:
df_q5 = df.na.fill("Unknown", subset=["status"])

df_q5.select("status").show()

+--------+
|  status|
+--------+
|  Active|
|  Active|
| Unknown|
|  Active|
|Inactive|
|  Active|
|  Active|
|  Active|
|Inactive|
|  Active|
+--------+



### Insight

Missing status values were replaced with 'Unknown' instead of removing records, ensuring data is retained for analysis.

## Q6. Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [9]:
from pyspark.sql.functions import count

df.groupBy("city") \
  .agg(count("*").alias("record_count")) \
  .filter("record_count > 100") \
  .show()

+----+------------+
|city|record_count|
+----+------------+
+----+------------+



### Insight

This query identifies cities with a high volume of records. In the current sample dataset, no city may satisfy the condition because the dataset is small.

## Q7. Explain why Spark DataFrames are immutable.

Spark DataFrames are immutable, meaning their data cannot be modified after creation. Any transformation such as filter(), select(), or withColumn() creates a new DataFrame instead of changing the existing one. This improves fault tolerance, consistency, and parallel processing.

## Q8. Modify the schema by converting raw_timestamp from string to timestamp and rename the column to event_timestamp.

In [10]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

df_q8 = df.withColumn(
    "event_timestamp",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

df_q8.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)



### Insight

The timestamp column was converted to a proper timestamp datatype, making it suitable for time-based analysis and processing.

## Q9. Remove rows where email or username is null.

In [11]:
df_q9 = df.na.drop(subset=["email", "username"])

df_q9.show()

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|         Fashion|        300| Delhi| 19|       Basic|    NULL|300.0|       1|user2@gmail.com|   priya|2025-01-02 11:00:00|
|    103|      2025-01-03|  West|     Electronics|        700|Mu

In [12]:
df_q9.count()

8

### Insight

Rows with missing email or username values were removed to ensure data quality and completeness.

## Q10. Build a complete data processing pipeline that removes duplicates, fills missing values, and calculates total revenue by store_id.

In [13]:
from pyspark.sql.functions import sum

pipeline_df = (
    df.dropDuplicates()
      .na.fill("Unknown")
      .groupBy("store_id")
      .agg(sum("sale_amount").alias("total_revenue"))
)

pipeline_df.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       1|         1600|
|       3|          900|
|       2|         1150|
+--------+-------------+



In [14]:
pipeline_df.orderBy("store_id").show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       1|         1600|
|       2|         1150|
|       3|          900|
+--------+-------------+



### Insight

The pipeline combines data cleaning and aggregation in a single workflow. It removes duplicates, handles missing values, and calculates total revenue generated by each store.

## Q11. Explain what a shuffle operation is in Spark and why it can impact performance.

A shuffle operation occurs when Spark redistributes data across partitions during operations such as groupBy(), join(), distinct(), and orderBy(). Shuffling involves data movement between executors, which increases network and disk I/O. Since data transfer is expensive, excessive shuffle operations can significantly reduce Spark performance.

## Q12. What is a wide transformation? Give examples.

A wide transformation is a Spark transformation that requires data movement between partitions. These transformations trigger shuffle operations. Examples include groupBy(), join(), distinct(), repartition(), and orderBy(). Wide transformations are generally more expensive than narrow transformations.

## Q13. Why is handling inconsistent data important before aggregation and analysis?

Inconsistent data such as null values, duplicate records, incorrect data types, and formatting differences can lead to inaccurate analysis results. Cleaning and standardizing data before aggregation ensures reliable insights and prevents genuine records from being excluded due to inconsistencies.

## Q14. Why is schema management important in Spark DataFrames?

Schema management ensures that columns have correct data types and names. Proper schemas improve query performance, reduce processing errors, enable accurate calculations, and support efficient data transformations and analytics.

## Q15. Summarize the complete data processing workflow performed in this assignment.

The assignment involved loading data into Spark DataFrames, understanding Spark fundamentals, removing duplicate records, handling missing values, filtering data, performing aggregations, modifying schemas, and building a complete processing pipeline. These operations demonstrated how Spark can efficiently clean, transform, and analyze data for large-scale processing.

# Conclusion

This assignment demonstrated the use of Apache Spark DataFrames for data cleaning, transformation, aggregation, schema management, and pipeline creation. Spark's in-memory processing and distributed architecture make it a powerful framework for large-scale data processing and analytics.